In [1]:
%load_ext autoreload
%autoreload 2
import os
import imageio.v3 as iio
import json
from IPython.display import Video

# INPUTS
n_drones = 500
video_name = "trex"
user_prompt = "A T-Rex roaring and turning its head from left to right standing on a large flat rock."

# SETUP OUTPUT PATHS
output_base_dir = f"/app/out/{video_name}_{n_drones}_drones/"
# ---------------------- Stage 1: Video Generation ----------------------
os.makedirs(output_base_dir + "/video_gen/", exist_ok=True)
prompt_output_path = output_base_dir + "/video_gen/prompt.json"
image_output_path = output_base_dir + "/video_gen/image.png"
video_output_path = output_base_dir + "/video_gen/video.mp4"
# ---------------------- Stage 2: Tracking ------------------------------
tracking_output_dir = output_base_dir + "/tracking/"
os.makedirs(tracking_output_dir, exist_ok=True)
# ---------------------- Stage 3: Trajectory Generation -----------------
trajectory_output_dir = output_base_dir + "/trajectory_generation/"
os.makedirs(trajectory_output_dir, exist_ok=True)
# ---------------------- Stage 4: Simulation and Safety Filter ----------
simulation_output_dir = output_base_dir + "/simulation/"
os.makedirs(simulation_output_dir, exist_ok=True)

In [ ]:
from swan.video_generation import VideoGenerationConfig, generate_video

# video generation
await generate_video(config=VideoGenerationConfig(), user_prompt=user_prompt, image_output_path=image_output_path, video_output_path=video_output_path, prompt_output_path=prompt_output_path)

In [2]:
from swan.tracking import TrackingConfig, track_video

# Optionally, you can supply a custom video + segmentation prompt manually and skip stage 1.
video = iio.imread(video_output_path)
segmentation_prompt = json.load(open(prompt_output_path))["segmentation_prompt"]

visualization_path = track_video(
    tracking_config=TrackingConfig(n_simultaneous_tracking_points=n_drones), # by default equal to n_drones can be adjusted.
    segmentation_prompt=segmentation_prompt, 
    video=video, 
    output_dir=tracking_output_dir, 
)
Video(visualization_path)

Attempting to load processor from local path: /weights/grounding-dino-base/
Attempting to load model from local path: /weights/grounding-dino-base/


Loading weights:   0%|          | 0/1206 [00:00<?, ?it/s]

/opt/venv/lib/python3.12/site-packages/torchvision/transforms/functional.py:154: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  img = torch.from_numpy(pic.transpose((2, 0, 1))).contiguous()



































































































































































Saved segmentation results to /app/out/trex_500_drones//tracking/segmentation_results.npz, size: 0.60 MB


Tracking Frames:  99%|█████████▉| 80/81 [04:06<00:03,  3.08s/it]


Saving tajectory data to /app/out/trex_500_drones//tracking/


In [3]:
from swan.trajectory_generation import TrajectoryGenerationConfig, generate_trajectories

trajectory_generation_config = TrajectoryGenerationConfig()
trajectory_splines_final, t_frames_final, n_required_extra_drones, transformation_matrix = \
    generate_trajectories(
        config=trajectory_generation_config,
        input_dir=tracking_output_dir,
        output_dir=trajectory_output_dir)
if n_required_extra_drones > 0:
    print(f"Warning: {n_required_extra_drones} extra drones are required to execute the trajectories safely. Rerun tracking with a smaller n_simultaneous_tracking_points or adjust the safety parameters in the config.")

Building bipartite graph...: 100%|██████████| 54/54 [00:03<00:00, 14.41it/s]


Total potential transitions checked: 4917
Transitions denied by fast check: 1683
Transitions denied after full check: 1239
Transitions allowed: 1995
Number of required extra drones: -6
54 transition tasks created.


Routing transitions...: 100%|██████████| 54/54 [00:09<00:00,  5.88it/s]


Total show duration: 168.28 seconds


In [ ]:
from swan.simulation import SimulationConfig, simulate_with_safety_filter

simulate_with_safety_filter(
    output_simulation_dir=simulation_output_dir,
    input_trajectory_dir=trajectory_output_dir,
    simulation_config=SimulationConfig(trajectory_generation_config=trajectory_generation_config, return_dense_trajectories=True)
)

Failed to import warp: No module named 'warp'
Failed to import mujoco_warp: No module named 'warp'
Time horizon: 6.25 s
Total show duration: 168.28 seconds with 215 evaluation points for MPC waypoints.


Simulating:  32%|███▏      | 433/1346 [06:43<27:53,  1.83s/it, MPC Success Rate=98.2%] 

In [ ]:
from swan.simulation import generate_visualization_video

# visualize simulated trajectories
generate_visualization_video(
    simulation_dir=simulation_output_dir,
    trajectory_generation_dir=trajectory_output_dir,
    video_path=video_output_path,
    background_color=(0, 0, 0),
    point_radius=5
)